In [9]:
import os

In [10]:
os.chdir("/Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC")

In [11]:
%pwd

'/Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC'

In [13]:
import mlflow

In [14]:
import dagshub
dagshub.init(repo_owner='Nihal7778', repo_name='End-End-Kidney-classification-using-MLflow-and-DVC', mlflow=True)

os.environ['MLFLOW_TRACKING_USERNAME'] = 'Nihal7778'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '67563d147cac72c014987f56508cb92e23f67223'

# Set tracking URI
mlflow.set_tracking_uri("https://dagshub.com/Nihal7778/End-End-Kidney-classification-using-MLflow-and-DVC.mlflow")

# Create or get experiment
experiment_name = "kidney-classification"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
except:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_name)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Initialized MLflow to track repo "Nihal7778/End-End-Kidney-classification-using-MLflow-and-DVC"

Repository Nihal7778/End-End-Kidney-classification-using-MLflow-and-DVC initialized!

2026/02/28 19:32:54 INFO mlflow.tracking._tracking_service.client: 🏃 View run valuable-sheep-656 at: https://dagshub.com/Nihal7778/End-End-Kidney-classification-using-MLflow-and-DVC.mlflow/#/experiments/1/runs/8ea6610ddbbf440c8268a4b2515c20c9.
2026/02/28 19:32:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/Nihal7778/End-End-Kidney-classification-using-MLflow-and-DVC.mlflow/#/experiments/1.


In [15]:
import tensorflow as tf
tf.keras.__version__ = tf.__version__

In [16]:
model = tf.keras.models.load_model("artifacts/training/trained_model.h5")

In [17]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_url: str
    params_image_size: list
    params_batch_size: int

In [18]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

Config path: /Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC/config/config.yaml
Config exists: True


In [25]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH, 
        params_filepath=PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/trained_model.h5",
            training_data="artifacts/data_ingestion/unzip/Chicken-fecal-images",
            mlflow_url="https://dagshub.com/Nihal7778/End-End-Kidney-classification-using-MLflow-and-DVC.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config
    

In [26]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse



In [28]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)  

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_url)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme  
        mlflow.log_params(self.config.all_params)
        mlflow.log_metrics({"loss": self.score[0], "accuracy": self.score[1]})

        if tracking_url_type_store != "file":
            mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16MODEL")  
        else:
            mlflow.keras.log_model(self.model, "model")

In [29]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
    raise e


[2026-02-28 19:43:47,702: INFO: common]: yaml file: /Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC/config/config.yaml loaded successfully]
[2026-02-28 19:43:47,705: INFO: common]: yaml file: /Users/nihal/Downloads/End-End-Kidney-classification-using-MLflow-and-DVC/params.yaml loaded successfully]
[2026-02-28 19:43:47,706: INFO: common]: created directory at: artifacts]
Found 116 images belonging to 2 classes.
8/8 [==============================] - 11s 1s/step - loss: 0.3785 - accuracy: 0.9569


2026/02/28 19:43:59 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: /var/folders/g5/54vbnb5j1gdc46kk4fr0tvmr0000gn/T/tmpz9pfvlfd/model/data/model/assets
[2026-02-28 19:44:00,165: INFO: builder_impl]: Assets written to: /var/folders/g5/54vbnb5j1gdc46kk4fr0tvmr0000gn/T/tmpz9pfvlfd/model/data/model/assets]


2026/02/28 19:44:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'VGG16MODEL'.
2026/02/28 19:44:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16MODEL, version 1
Created version '1' of model 'VGG16MODEL'.


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Set alias instead of stage
client.set_registered_model_alias(
    name="VGG16MODEL",
    alias="staging",
    version="1"
)